# Component 2 — Tooth ROI detector + crop classifier (v1)

This notebook runs the full five-fold Colab GPU experiment from the prepared allowlisted package. Its image-level MGI labels are provisional caption-derived grades. Tooth-level detection and crop classification are measured separately; neither is a diagnosis. The model takes the maximum predicted grade among detected tooth ROIs for image-level MGI. A no-detection case is counted as an incorrect abstention.

The package includes only 680 frozen Training images and 53 previously selected Validation images. The primary five-fold experiment uses only the 680 Training records. The 53 Validation records are not used for selection or primary evaluation. The 181 held records and protected Test payloads are absent. Do not add them.

Before running, upload `roi_tooth_dataset_v1.zip` to the Drive location below. This notebook saves checkpoints to Drive each epoch and can resume when rerun with the same configuration and accelerator. Keep the same output directory and configuration when resuming. Use a new versioned output directory for any changed settings.

The pinned Ultralytics 8.4.178 package is marked AGPL-3.0 in the installed package metadata. Treat this as a research prototype and review the model/library license before distributing a product.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Upload the prepared ZIP to this Drive folder before running this cell.
DRIVE_ROOT = Path('/content/drive/MyDrive/RP-J26-IT-458/roi_tooth_v1')
PACKAGE_ZIP = DRIVE_ROOT / 'roi_tooth_dataset_v1_oneclass_v4.zip'
EXPECTED_ZIP_SHA256 = '9b1e797f7f593438a5318410ed2aa1cfdd7a66b8db0aac865b3229b63d3a20c8'
PACKAGE_EXTRACT_ROOT = Path('/content/roi_tooth_package')
PACKAGE_ROOT = PACKAGE_EXTRACT_ROOT / 'roi_tooth_v1'
OUTPUT_ROOT = DRIVE_ROOT / 'colab_run_v1'

print('Package:', PACKAGE_ZIP)
print('Output:', OUTPUT_ROOT)
assert PACKAGE_ZIP.is_file(), 'Copy the ZIP to the Drive path above first.'


In [ ]:
import hashlib
import json
import zipfile
from pathlib import PurePosixPath

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

actual_zip_sha256 = sha256_file(PACKAGE_ZIP)
print('ZIP SHA-256:', actual_zip_sha256)
assert actual_zip_sha256 == EXPECTED_ZIP_SHA256, 'ZIP checksum does not match the local handoff.'

if not PACKAGE_ROOT.is_dir():
    PACKAGE_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PACKAGE_ZIP) as archive:
        members = archive.namelist()
        assert all(PurePosixPath(name).parts[0] == 'roi_tooth_v1' for name in members)
        assert all('..' not in PurePosixPath(name).parts for name in members)
        bad_member = archive.testzip()
        assert bad_member is None, f'ZIP CRC check failed at {bad_member}'
        archive.extractall(PACKAGE_EXTRACT_ROOT)

required = [
    PACKAGE_ROOT / 'image_manifest.json', PACKAGE_ROOT / 'folds.json',
    PACKAGE_ROOT / 'crop_manifest.csv', PACKAGE_ROOT / 'images',
    PACKAGE_ROOT / 'labels', PACKAGE_ROOT / 'source_annotations',
    PACKAGE_ROOT / 'code' / 'roi_colab_train_v1.py',
]
assert all(path.exists() for path in required), 'Extracted package is incomplete.'
print('ZIP and package structure verified.')


## Validate the allowlist and derived annotations

This cell checks every image and copied source TXT against the per-record SHA-256 stored in the package manifest. It also verifies each derived detector row has one of five classes (0–4), valid normalized coordinates, and the saved five-fold Training IDs cover all 680 allowed records exactly once. These checks do not clinically verify the labels.


In [ ]:
import csv

manifest = json.loads((PACKAGE_ROOT / 'image_manifest.json').read_text())
fold_data = json.loads((PACKAGE_ROOT / 'folds.json').read_text())
records = manifest['records']
assert len(records) == 733
assert sum(row['original_split'] == 'Training' for row in records) == 680
assert sum(row['original_split'] == 'Validation' for row in records) == 53
assert all(row['label_status'] == 'provisional_not_clinically_verified' for row in records)
assert all(not row['qualified_label_approved'] for row in records)
assert all('Test' not in row['source_record_id'] for row in records)
assert manifest['derived_detector_classes'] == {'0': 'tooth_roi'}
assert manifest['source_class_mapping'] == {str(raw): raw - 2 for raw in range(2, 7)}
assert manifest['selected_tooth_roi_boxes_and_crops'] == 8796
assert len(fold_data['folds']) == 5

training_ids = {row['source_record_id'] for row in records if row['original_split'] == 'Training'}
scored_ids = [record_id for fold in fold_data['folds'] for record_id in fold['score_ids']]
assert set(scored_ids) == training_ids and len(scored_ids) == len(set(scored_ids)) == 680
record_by_id = {row['source_record_id']: row for row in records}
for fold in fold_data['folds']:
    fit_ids, score_ids = set(fold['fit_ids']), set(fold['score_ids'])
    assert fit_ids.isdisjoint(score_ids)
    fit_groups = {record_by_id[item]['content_group_id'] for item in fit_ids}
    score_groups = {record_by_id[item]['content_group_id'] for item in score_ids}
    assert fit_groups.isdisjoint(score_groups)

label_rows = 0
raw_grade_counts = {grade: 0 for grade in range(5)}
for row in records:
    source_id = row['source_record_id']
    split, image_id = source_id.split('/')
    image_path = PACKAGE_ROOT / row['model_image_path']
    source_label_path = PACKAGE_ROOT / 'source_annotations' / split / f'{image_id}.txt'
    derived_label_path = PACKAGE_ROOT / 'labels' / split / f'{image_id}.txt'
    assert sha256_file(image_path) == row['model_image_sha256'], source_id
    assert sha256_file(source_label_path) == row['source_label_sha256'], source_id
    source_lines = [line.split() for line in source_label_path.read_text().splitlines() if line.strip()]
    tooth_source_classes = [int(line[0]) for line in source_lines if int(line[0]) in range(2, 7)]
    assert len(tooth_source_classes) == row['tooth_box_count']
    assert max(class_id - 2 for class_id in tooth_source_classes) == row['provisional_image_mgi_grade']
    for class_id in tooth_source_classes:
        raw_grade_counts[class_id - 2] += 1
    derived_lines = [line.split() for line in derived_label_path.read_text().splitlines() if line.strip()]
    assert len(derived_lines) == row['tooth_box_count']
    assert all(len(line) == 5 and int(line[0]) == 0 for line in derived_lines)
    for line in derived_lines:
        assert all(0 <= float(value) <= 1 for value in line[1:])
    label_rows += len(derived_lines)

with (PACKAGE_ROOT / 'crop_manifest.csv').open(encoding='utf-8', newline='') as stream:
    crop_rows = list(csv.DictReader(stream))
assert len(crop_rows) == 8796
assert all((PACKAGE_ROOT / row['crop_path']).is_file() for row in crop_rows)
print({'images': len(records), 'Training': len(training_ids), 'Validation': 53,
       'Training folds': 5, 'one-class detector boxes': label_rows,
       'crop grade counts 0-4': raw_grade_counts, 'crops': len(crop_rows),
       'protected Test images included': 0})


In [ ]:
%pip install -q -r /content/roi_tooth_package/roi_tooth_v1/code/requirements-vision-roi-v1.txt


In [ ]:
import sys
import torch, torchvision, ultralytics

CODE_ROOT = PACKAGE_ROOT / 'code'
sys.path.insert(0, str(CODE_ROOT))
from roi_colab_train_v1 import run_experiment

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))


## Run the fixed five-fold experiment

The one-class detector is trained only to find tooth/perigingival ROIs (derived detector class 0 means tooth_roi, not MGI 0). The separate ResNet-18 crop classifier maps detected boxes to MGI grades 0–4 using the audited original source IDs 2–6. It is fitted only on crops from each fold's fit images; class weights are also calculated from those fit crops only. Detector mAP uses the held-out fold's boxes. Image-level MGI is computed separately as the maximum crop-classifier grade among predicted boxes. No validation-set checkpoint selection is used; the fixed final epoch is evaluated.

If Colab disconnects, rerun the same cells with the same Drive output folder and settings. The detector resumes from Ultralytics `last.pt`; the crop classifier resumes from `checkpoint_last.pt`. Do not change the epoch count or accelerator in that output folder.


In [ ]:
DETECTOR_EPOCHS = 30
CROP_CLASSIFIER_EPOCHS = 12
DETECTOR_BATCH = 16
CROP_BATCH = 64
WORKERS = 2
SEED = 42

run_result = run_experiment(
    package_root=PACKAGE_ROOT,
    output_root=OUTPUT_ROOT,
    device='0',
    detector_epochs=DETECTOR_EPOCHS,
    classifier_epochs=CROP_CLASSIFIER_EPOCHS,
    detector_batch=DETECTOR_BATCH,
    crop_batch=CROP_BATCH,
    workers=WORKERS,
    seed=SEED,
    detector_weights='yolo26n.pt',
    folds=[0, 1, 2, 3, 4],
)
print(run_result['status'], run_result['records'])


In [ ]:
oof_metrics = json.loads((OUTPUT_ROOT / 'oof_metrics.json').read_text())
print(json.dumps({key: value for key, value in oof_metrics.items() if key != 'confusion_matrix'}, indent=2))
print('Confusion matrix labels:', oof_metrics['confusion_matrix_labels'])
print('Confusion matrix:', oof_metrics['confusion_matrix'])
assert oof_metrics['status'] == 'complete_680_image_oof'
assert oof_metrics['n_images'] == 680


## Package checkpoints and evidence for return to the Mac

This final cell copies only this experiment's five detector checkpoints, five crop-classifier checkpoints, OOF predictions, metrics and run configuration into a separate return ZIP. It does not replace the existing PP1 model. Download `roi_tooth_v1_colab_return.zip` from the Drive folder and place its contents under the new ignored local directory `data/vision/roi_tooth_v1_20261017/colab_return/`. Verify the included SHA-256 manifest before using the isolated Mac demo adapter.


In [ ]:
import shutil

RETURN_ROOT = OUTPUT_ROOT / 'return_bundle'
RETURN_ZIP = DRIVE_ROOT / 'roi_tooth_v1_colab_return.zip'
if RETURN_ROOT.exists() or RETURN_ZIP.exists():
    if not RETURN_ROOT.is_dir() or not RETURN_ZIP.is_file():
        raise FileExistsError('A partial return bundle exists. Preserve it and use a new versioned output folder.')
    with zipfile.ZipFile(RETURN_ZIP) as archive:
        bad_member = archive.testzip()
        assert bad_member is None, f'Existing return ZIP CRC failed at {bad_member}'
    print('Preserved existing return bundle:', RETURN_ZIP)
    print('ZIP SHA-256:', sha256_file(RETURN_ZIP))
else:
    RETURN_ROOT.mkdir(parents=True)
    for fold in range(5):
        fold_root = OUTPUT_ROOT / f'fold_{fold:02d}'
        target = RETURN_ROOT / f'fold_{fold:02d}'
        target.mkdir(parents=True)
        shutil.copy2(fold_root / 'detector' / 'detector_run' / 'weights' / 'last.pt', target / 'detector_last.pt')
        shutil.copy2(fold_root / 'roi_classifier' / 'checkpoint_last.pt', target / 'crop_classifier_last.pt')
        shutil.copy2(fold_root / 'fold_metrics.json', target / 'fold_metrics.json')
    for name in ['run_configuration.json', 'oof_metrics.json', 'oof_predictions.csv', 'experiment_complete.json']:
        shutil.copy2(OUTPUT_ROOT / name, RETURN_ROOT / name)

    hash_rows = []
    for path in sorted(RETURN_ROOT.rglob('*')):
        if path.is_file() and path.name != 'return_integrity_manifest.csv':
            hash_rows.append({'path': str(path.relative_to(RETURN_ROOT)), 'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
    with (RETURN_ROOT / 'return_integrity_manifest.csv').open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['path', 'bytes', 'sha256'])
        writer.writeheader()
        writer.writerows(hash_rows)
    with zipfile.ZipFile(RETURN_ZIP, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(RETURN_ROOT.rglob('*')):
            if path.is_file():
                archive.write(path, Path('roi_tooth_v1_colab_return') / path.relative_to(RETURN_ROOT))
    print('Return ZIP:', RETURN_ZIP)
    print('ZIP SHA-256:', sha256_file(RETURN_ZIP))
    print('Checkpoints included:', 10)
